# 2D Infection Simulation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ryouy/simulations/blob/main/infection_simulation.ipynb)

30個の球が2次元空間を動き、赤い球との接触で感染が広がるシミュレーションです。上から順番に実行してください。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
from pathlib import Path
import csv, json

N = 30
SEED = 42
STEPS = 600
DT = 0.025
RADIUS = 0.018
SPEED = 0.22
TURN_NOISE = 0.35
rng = np.random.default_rng(SEED)
out = Path("outputs")
out.mkdir(exist_ok=True)

# Avoid overlapping initial positions.
positions = []
while len(positions) < N:
    p = rng.uniform(RADIUS, 1-RADIUS, 2)
    if all(np.linalg.norm(p-q) >= 2.2*RADIUS for q in positions):
        positions.append(p)
pos = np.array(positions)
angles = rng.uniform(0, 2*np.pi, N)
vel = SPEED * np.column_stack((np.cos(angles), np.sin(angles)))
infected = np.zeros(N, dtype=bool)
infected[0] = True
frames = [pos.copy()]
states = [infected.copy()]
counts = [1]

for step in range(STEPS):
    angles = np.arctan2(vel[:,1], vel[:,0]) + rng.normal(0, TURN_NOISE*np.sqrt(DT), N)
    vel = SPEED*np.column_stack((np.cos(angles), np.sin(angles)))
    pos += vel*DT
    for axis in (0,1):
        low = pos[:,axis] < RADIUS
        high = pos[:,axis] > 1-RADIUS
        pos[low,axis] = 2*RADIUS-pos[low,axis]
        pos[high,axis] = 2*(1-RADIUS)-pos[high,axis]
        vel[low|high,axis] *= -1
    for i in range(N):
        for j in range(i+1,N):
            delta = pos[j]-pos[i]
            dist = np.linalg.norm(delta)
            if dist < 2*RADIUS:
                if infected[i] or infected[j]:
                    infected[i] = infected[j] = True
                normal = delta/dist if dist > 1e-12 else np.array([1.,0.])
                overlap = 2*RADIUS-dist
                pos[i] -= normal*overlap/2
                pos[j] += normal*overlap/2
                relative = np.dot(vel[i]-vel[j], normal)
                if relative > 0:
                    vel[i] -= relative*normal
                    vel[j] += relative*normal
    pos = np.clip(pos, RADIUS, 1-RADIUS)
    frames.append(pos.copy())
    states.append(infected.copy())
    counts.append(int(infected.sum()))
print(f"Final infected: {counts[-1]}/{N}")

In [ ]:
fig, ax = plt.subplots(figsize=(6,6), facecolor="#111111")
ax.set_facecolor("#111111")
ax.set_xlim(0,1); ax.set_ylim(0,1); ax.set_aspect("equal")
ax.set_xticks([]); ax.set_yticks([])
for spine in ax.spines.values(): spine.set_color("#777777")
scatter = ax.scatter([], [], s=160, edgecolors="#999999", linewidths=0.6)
label = ax.text(0.02,1.02,"",transform=ax.transAxes,color="white")
def draw(k):
    scatter.set_offsets(frames[k])
    scatter.set_facecolors(np.where(states[k], "#e53935", "#ffffff"))
    label.set_text(f"Frame {k}  |  Infected {counts[k]}/{N}")
    return scatter, label
ani = FuncAnimation(fig, draw, frames=range(0,len(frames),3), interval=40, blit=True)
plt.close(fig)
display(HTML(ani.to_jshtml()))

In [ ]:
with (out/"infection_counts.csv").open("w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["step","time","infected"])
    writer.writerows((i, i*DT, c) for i,c in enumerate(counts))
(out/"settings.json").write_text(json.dumps(dict(N=N, SEED=SEED, STEPS=STEPS, DT=DT, RADIUS=RADIUS, SPEED=SPEED, TURN_NOISE=TURN_NOISE), indent=2))
fig2, ax2 = plt.subplots()
ax2.plot(np.arange(len(counts))*DT, counts)
ax2.set(xlabel="Time",ylabel="Infected",ylim=(0,N+1),title="Infection growth")
fig2.savefig(out/"infection_curve.png",dpi=180,bbox_inches="tight")
plt.show()
# MP4 requires ffmpeg (normally available in Colab).
try:
    from matplotlib.animation import FFMpegWriter
    ani.save(str(out/"infection.mp4"),writer=FFMpegWriter(fps=25),dpi=110)
    print("Saved MP4")
except Exception as e:
    print("MP4 export unavailable:",e)
from matplotlib.animation import PillowWriter
ani.save(str(out/"infection.gif"),writer=PillowWriter(fps=15),dpi=85)
print("Saved:",[p.name for p in out.iterdir()])

## GitHubに結果を保存（任意）
Colab左側の鍵アイコン（Secrets）で `GITHUB_TOKEN` を登録し、「ノートブックからのアクセス」を許可してください。Fine-grained tokenの場合、対象リポジトリの **Contents: Read and write** が必要です。トークンをノートブックに直接書かないでください。

In [ ]:
from google.colab import userdata
import requests, base64
TOKEN = userdata.get("GITHUB_TOKEN")
if not TOKEN:
    raise ValueError("Colab Secretsに GITHUB_TOKEN を設定してください")
REPO = "ryouy/simulations"
headers = {"Authorization": f"Bearer {TOKEN}", "Accept": "application/vnd.github+json", "X-GitHub-Api-Version":"2022-11-28"}
for file in out.iterdir():
    path = f"outputs/{file.name}"
    url = f"https://api.github.com/repos/{REPO}/contents/{path}"
    existing = requests.get(url,headers=headers,timeout=30)
    if existing.status_code not in (200,404):
        existing.raise_for_status()
    payload = {"message":f"Save simulation output: {file.name}", "content":base64.b64encode(file.read_bytes()).decode(), "branch":"main"}
    if existing.status_code == 200:
        payload["sha"] = existing.json()["sha"]
    response = requests.put(url,headers=headers,json=payload,timeout=120)
    response.raise_for_status()
    print("Uploaded:",path)
print("Finished")